In [ ]:
import os
import shutil
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# 1. PATH CONFIGURATION (Matches your Drive structure)
DRIVE_IMG = "/content/drive/MyDrive/satellite data/extracting_slums_from_satellite_imagery/extracting_slums_from_satellite_imagery/images"
DRIVE_LBL = "/content/drive/MyDrive/satellite data/extracting_slums_from_satellite_imagery/extracting_slums_from_satellite_imagery/labels"

# 2. LOCAL SYNC (The "Pro" speed fix)
!mkdir -p /content/local_data/images /content/local_data/labels
print("Syncing data to local SSD...")
!cp -r "{DRIVE_IMG}/." /content/local_data/images/
!cp -r "{DRIVE_LBL}/." /content/local_data/labels/
print("Sync Complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Syncing data to local SSD...
Sync Complete.


In [ ]:
import glob
import os
from sklearn.model_selection import train_test_split

# Get all image and label paths
all_img_paths = sorted(glob.glob("/content/local_data/images/*.tif"))
all_lbl_paths = sorted(glob.glob("/content/local_data/labels/*.tif"))

# Extract base filenames (without path and extension)
img_basenames = {os.path.basename(p).split('.')[0] for p in all_img_paths}
lbl_basenames = {os.path.basename(p).split('.')[0] for p in all_lbl_paths}

# Find common basenames
common_basenames = sorted(list(img_basenames.intersection(lbl_basenames)))

# Filter paths to include only those with common basenames
all_img_files = []
all_lbl_files = []

for basename in common_basenames:
    img_path = f"/content/local_data/images/{basename}.tif"
    lbl_path = f"/content/local_data/labels/{basename}.tif"

    # Ensure both files actually exist before adding them
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        all_img_files.append(img_path)
        all_lbl_files.append(lbl_path)

# Now, all_img_files and all_lbl_files should have the same length
# Split: 80% Train, 20% for Val/Test
train_imgs, temp_imgs, train_lbls, temp_lbls = train_test_split(
    all_img_files, all_lbl_files, test_size=0.2, random_state=42
)

# Split the 20% into 10% Val and 10% Test
val_imgs, test_imgs, val_lbls, test_lbls = train_test_split(
    temp_imgs, temp_lbls, test_size=0.5, random_state=42
)

In [ ]:
print(f"Total images: {len(all_img_files)}")
print(f"Training images: {len(train_imgs)}")
print(f"Validation images: {len(val_imgs)}")
print(f"Test images: {len(test_imgs)}")

Total images: 2323
Training images: 1858
Validation images: 232
Test images: 233


In [ ]:
TILE_SIZE = 256
STRIDE = 256
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]


In [ ]:
import rasterio
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import SegformerForSemanticSegmentation
import torch.nn.functional as F


In [ ]:
def load_tiff_pair(img_path, lbl_path):
    with rasterio.open(img_path) as src:
        img = src.read()[:3]   # RGB only

    with rasterio.open(lbl_path) as src:
        mask = src.read(1)

    mask = (mask > 0).astype("uint8")  # force binary
    return img, mask


In [ ]:
def tile_image(img, mask):

    tiles = []

    for y in range(0, img.shape[1] - TILE_SIZE, STRIDE):

        for x in range(0, img.shape[2] - TILE_SIZE, STRIDE):

            img_t = img[:, y:y+TILE_SIZE, x:x+TILE_SIZE]

            mask_t = mask[y:y+TILE_SIZE, x:x+TILE_SIZE]

            tiles.append((img_t, mask_t))

    return tiles

In [ ]:
def build_tiles(img_list, lbl_list):
    tiles = []
    for img_p, lbl_p in zip(img_list, lbl_list):
        img, mask = load_tiff_pair(img_p, lbl_p)
        tiles.extend(tile_image(img, mask))
    return tiles

train_tiles = build_tiles(train_imgs, train_lbls)
val_tiles   = build_tiles(val_imgs, val_lbls)
test_tiles  = build_tiles(test_imgs, test_lbls)

print("Train tiles:", len(train_tiles))
print("Val tiles:", len(val_tiles))
print("Test tiles:", len(test_tiles))


Train tiles: 1858
Val tiles: 232
Test tiles: 233


In [ ]:
class SlumDataset(Dataset):

    def __init__(self, tiles):

        self.tiles = tiles



    def __len__(self):

        return len(self.tiles)



    def __getitem__(self, idx):

        img, mask = self.tiles[idx]



        img = img / 255.0

        img = (img - np.array(MEAN).reshape(3,1,1)) / np.array(STD).reshape(3,1,1)



        img = torch.tensor(img, dtype=torch.float32)

        mask = torch.tensor(mask, dtype=torch.long)



        return img, mask

In [ ]:
train_loader = DataLoader(SlumDataset(train_tiles), batch_size=8, shuffle=True)

val_loader   = DataLoader(SlumDataset(val_tiles), batch_size=8)

test_loader  = DataLoader(SlumDataset(test_tiles), batch_size=8)

In [ ]:
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512",
    num_labels=1,
    ignore_mismatched_sizes=True
)

model.to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/110M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/110M [00:00<?, ?B/s]

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b2-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([150, 768, 1, 1]) in the checkpoint and torch.Size([1, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)

In [ ]:

from tqdm.auto import tqdm



optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5)



EPOCHS = 30



for epoch in range(EPOCHS):

    model.train()

    train_loss = 0



    # Wrap train_loader with tqdm for a progress bar

    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} Training"):

        imgs, masks = imgs.to(device), masks.to(device)



        outputs = model(pixel_values=imgs, labels=masks)

        loss = outputs.loss



        optimizer.zero_grad()

        loss.backward()

        optimizer.step()



        train_loss += loss.item()



    model.eval()

    val_loss = 0

    with torch.no_grad():

        # Wrap val_loader with tqdm for a progress bar

        for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} Validation"):

            imgs, masks = imgs.to(device), masks.to(device)

            val_loss += model(pixel_values=imgs, labels=masks).loss.item()



    print(

        f"Epoch {epoch+1}/{EPOCHS} | "

        f"Train Loss: {train_loss/len(train_loader):.4f} | "

        f"Val Loss: {val_loss/len(val_loader):.4f}"

    )



torch.save(

    model.state_dict(),

    "/content/drive/MyDrive/segformer_slum.pth(2)"

)



print("Model saved to Drive.")

Epoch 1/30 Training:   0%|          | 0/233 [00:00<?, ?it/s]

Epoch 1/30 Validation:   0%|          | 0/29 [00:00<?, ?it/s]

Epoch 1/30 | Train Loss: 0.3134 | Val Loss: 0.1475


Epoch 2/30 Training:   0%|          | 0/233 [00:00<?, ?it/s]

Epoch 2/30 Validation:   0%|          | 0/29 [00:00<?, ?it/s]

...

Epoch 29/30 Validation:   0%|          | 0/29 [00:00<?, ?it/s]

Epoch 29/30 | Train Loss: 0.0215 | Val Loss: 0.0817


Epoch 30/30 Training:   0%|          | 0/233 [00:00<?, ?it/s]

Epoch 30/30 Validation:   0%|          | 0/29 [00:00<?, ?it/s]

Epoch 30/30 | Train Loss: 0.0226 | Val Loss: 0.0810
Model saved to Drive.


In [ ]:
def compute_iou(pred, mask):
    inter = (pred & mask).sum()
    union = (pred | mask).sum()
    return (inter / (union + 1e-6)).item()

model.eval()
ious = []

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs = imgs.to(device)
        logits = model(pixel_values=imgs).logits
        logits = F.interpolate(logits, size=(256,256))
        preds = (torch.sigmoid(logits) > 0.5).cpu().int()

        for p, m in zip(preds, masks):
            ious.append(compute_iou(p.squeeze(), m))

print("Mean Test IoU:", sum(ious) / len(ious))


Mean Test IoU: 0.43439567287336606


In [ ]:
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
from transformers import SegformerForSemanticSegmentation

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512",
    num_labels=1,
    ignore_mismatched_sizes=True
)

model.load_state_dict(
    torch.load("/content/drive/MyDrive/segformer_slum.pth(2)", map_location=device)
)

model.to(device)
model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/110M [00:00<?, ?B/s]

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b2-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([150, 768, 1, 1]) in the checkpoint and torch.Size([1, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/110M [00:00<?, ?B/s]

SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F
import numpy as np
import torch

def visualize_50_test_results(loader, model):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    count = 0
    max_images = 200

    # Standard normalization constants
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]


    for imgs, masks in loader:
        if count >= max_images:
            break

        imgs, masks = imgs.to(device), masks.to(device)

        with torch.no_grad():
            outputs = model(pixel_values=imgs)
            logits = outputs.logits
            # Rescale to 256x256
            logits = F.interpolate(logits, size=(256, 256), mode="bilinear", align_corners=False)
            # Binary prediction: White (1) for Slum, Black (0) for background
            preds = (torch.sigmoid(logits) > 0.5).float()

        batch_size = imgs.shape[0]

        for i in range(batch_size):
            if count >= max_images:
                break

            plt.figure(figsize=(15, 5))

            # 1. Original Image (Un-normalize for viewing)
            img_display = imgs[i].cpu().permute(1, 2, 0).numpy()
            img_display = img_display * std + mean
            img_display = np.clip(img_display, 0, 1)

            # 2. Ground Truth (Industry Standard: White=Slum, Black=Background)
            gt_display = masks[i].cpu().numpy()

            # 3. Prediction (Industry Standard: White=Slum, Black=Background)
            pred_display = preds[i].cpu().squeeze().numpy()

            # Plotting
            plt.subplot(1, 3, 1)
            plt.imshow(img_display)
            plt.title(f"Image {count+1}: Original")
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.imshow(gt_display, cmap='gray', vmin=0, vmax=1)
            plt.title("Ground Truth (White=Slum)")
            plt.axis('off')

            plt.subplot(1, 3, 3)
            plt.imshow(pred_display, cmap='gray', vmin=0, vmax=1)
            plt.title("Prediction (White=Slum)")
            plt.axis('off')

            plt.show()
            count += 1

# Execute the visualization
visualize_50_test_results(test_loader, model)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import glob
import os

def predict_multiple_custom_images(input_folder, model, device, threshold=0.5):
    # 1. Get all .tif files from the folder
    image_paths = sorted(glob.glob(os.path.join(input_folder, "*.tif")))

    if not image_paths:
        print(f"No .tif files found in {input_folder}")
        return

    model.eval()

    # Normalization constants
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

    for img_path in image_paths:
        print(f"Processing: {os.path.basename(img_path)}")

        # 2. Load Image
        with rasterio.open(img_path) as src:
            image = src.read()[:3]
            profile = src.profile

        # Prepare for plotting and model
        img_plot = np.transpose(image, (1, 2, 0)) / 255.0
        img_plot = np.clip(img_plot, 0, 1)
        img_tensor = (image / 255.0 - mean) / std

        # 3. Tiling Logic
        h, w = image.shape[1], image.shape[2]
        full_mask = np.zeros((h, w))

        with torch.no_grad():
            for y in range(0, h, 256):
                for x in range(0, w, 256):
                    tile = img_tensor[:, y:y+256, x:x+256]

                    t_h, t_w = tile.shape[1], tile.shape[2]
                    if t_h < 256 or t_w < 256:
                        tile = np.pad(tile, ((0,0), (0, 256-t_h), (0, 256-t_w)), mode='constant')

                    tile_torch = torch.from_numpy(tile).float().unsqueeze(0).to(device)

                    logits = model(pixel_values=tile_torch).logits
                    logits = F.interpolate(logits, size=(256, 256), mode="bilinear")
                    prob = torch.sigmoid(logits).squeeze().cpu().numpy()

                    full_mask[y:y+t_h, x:x+t_w] = prob[:t_h, :t_w]

        # 4. Create Binary Mask (White = Slum)
        binary_mask = (full_mask > threshold).astype(np.uint8)

        # 5. Visualize
        fig, ax = plt.subplots(1, 2, figsize=(12, 6))
        ax[0].imshow(img_plot)
        ax[0].set_title(f"Image: {os.path.basename(img_path)}")
        ax[0].axis('off')

        ax[1].imshow(binary_mask, cmap='gray', vmin=0, vmax=1)
        ax[1].set_title("Predicted Slum (White)")
        ax[1].axis('off')

        plt.tight_layout()
        plt.show()

INPUT_FOLDER_PATH = "/content/drive/MyDrive/inputs"

# Make sure the directory exists (helpful if running in Colab)
if not os.path.exists(INPUT_FOLDER_PATH):
    os.makedirs(INPUT_FOLDER_PATH)
    print(f"Please upload your .tif images to {INPUT_FOLDER_PATH} and run this cell again.")
else:
    predict_multiple_custom_images(INPUT_FOLDER_PATH, model, device, threshold=0.5)


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import glob
import os

def predict_multiple_custom_images(input_folder, model, device, threshold=0.5):
    # 1. Get all .tif files from the folder
    image_paths = sorted(glob.glob(os.path.join(input_folder, "*.tif")))

    if not image_paths:
        print(f"No .tif files found in {input_folder}")
        return

    model.eval()

    # Normalization constants
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

    for img_path in image_paths:
        print(f"Processing: {os.path.basename(img_path)}")

        # 2. Load Image
        with rasterio.open(img_path) as src:
            image = src.read()[:3]
            profile = src.profile

        # Prepare for plotting and model
        img_plot = np.transpose(image, (1, 2, 0)) / 255.0
        img_plot = np.clip(img_plot, 0, 1)
        img_tensor = (image / 255.0 - mean) / std

        # 3. Tiling Logic
        h, w = image.shape[1], image.shape[2]
        full_mask = np.zeros((h, w))

        with torch.no_grad():
            for y in range(0, h, 256):
                for x in range(0, w, 256):
                    tile = img_tensor[:, y:y+256, x:x+256]

                    t_h, t_w = tile.shape[1], tile.shape[2]
                    if t_h < 256 or t_w < 256:
                        tile = np.pad(tile, ((0,0), (0, 256-t_h), (0, 256-t_w)), mode='constant')

                    tile_torch = torch.from_numpy(tile).float().unsqueeze(0).to(device)

                    logits = model(pixel_values=tile_torch).logits
                    logits = F.interpolate(logits, size=(256, 256), mode="bilinear")
                    prob = torch.sigmoid(logits).squeeze().cpu().numpy()

                    full_mask[y:y+t_h, x:x+t_w] = prob[:t_h, :t_w]

        # 4. Create Binary Mask (White = Slum)
        binary_mask = (full_mask > threshold).astype(np.uint8)

        # 5. Visualize
        fig, ax = plt.subplots(1, 2, figsize=(12, 6))
        ax[0].imshow(img_plot)
        ax[0].set_title(f"Image: {os.path.basename(img_path)}")
        ax[0].axis('off')

        ax[1].imshow(binary_mask, cmap='gray', vmin=0, vmax=1)
        ax[1].set_title("Predicted Slum (White)")
        ax[1].axis('off')

        plt.tight_layout()
        plt.show()

INPUT_FOLDER_PATH = "/content/drive/MyDrive/inputs"

# Make sure the directory exists (helpful if running in Colab)
if not os.path.exists(INPUT_FOLDER_PATH):
    os.makedirs(INPUT_FOLDER_PATH)
    print(f"Please upload your .tif images to {INPUT_FOLDER_PATH} and run this cell again.")
else:
    predict_multiple_custom_images(INPUT_FOLDER_PATH, model, device, threshold=0.5)


In [ ]:
import torch
from transformers import SegformerForSemanticSegmentation

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512",
    num_labels=1,
    ignore_mismatched_sizes=True
)

model.load_state_dict(
    torch.load("/content/drive/MyDrive/segformer_slum.pth", map_location=device)
)

model.to(device)
model.eval()


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b2-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([150, 768, 1, 1]) in the checkpoint and torch.Size([1, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import glob
import os

def predict_multiple_custom_images(input_folder, model, device, threshold=0.5):
    # 1. Get all .tif files from the folder
    image_paths = sorted(glob.glob(os.path.join(input_folder, "*.tif")))

    if not image_paths:
        print(f"No .tif files found in {input_folder}")
        return

    model.eval()

    # Normalization constants
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

    for img_path in image_paths:
        print(f"Processing: {os.path.basename(img_path)}")

        # 2. Load Image
        with rasterio.open(img_path) as src:
            image = src.read()[:3]
            profile = src.profile

        # Prepare for plotting and model
        img_plot = np.transpose(image, (1, 2, 0)) / 255.0
        img_plot = np.clip(img_plot, 0, 1)
        img_tensor = (image / 255.0 - mean) / std

        # 3. Tiling Logic
        h, w = image.shape[1], image.shape[2]
        full_mask = np.zeros((h, w))

        with torch.no_grad():
            for y in range(0, h, 256):
                for x in range(0, w, 256):
                    tile = img_tensor[:, y:y+256, x:x+256]

                    t_h, t_w = tile.shape[1], tile.shape[2]
                    if t_h < 256 or t_w < 256:
                        tile = np.pad(tile, ((0,0), (0, 256-t_h), (0, 256-t_w)), mode='constant')

                    tile_torch = torch.from_numpy(tile).float().unsqueeze(0).to(device)

                    logits = model(pixel_values=tile_torch).logits
                    logits = F.interpolate(logits, size=(256, 256), mode="bilinear")
                    prob = torch.sigmoid(logits).squeeze().cpu().numpy()

                    full_mask[y:y+t_h, x:x+t_w] = prob[:t_h, :t_w]

        # 4. Create Binary Mask (White = Slum)
        binary_mask = (full_mask > threshold).astype(np.uint8)

        # 5. Visualize
        fig, ax = plt.subplots(1, 2, figsize=(12, 6))
        ax[0].imshow(img_plot)
        ax[0].set_title(f"Image: {os.path.basename(img_path)}")
        ax[0].axis('off')

        ax[1].imshow(binary_mask, cmap='gray', vmin=0, vmax=1)
        ax[1].set_title("Predicted Slum (White)")
        ax[1].axis('off')

        plt.tight_layout()
        plt.show()

INPUT_FOLDER_PATH = "/content/drive/MyDrive/inputs"

# Make sure the directory exists (helpful if running in Colab)
if not os.path.exists(INPUT_FOLDER_PATH):
    os.makedirs(INPUT_FOLDER_PATH)
    print(f"Please upload your .tif images to {INPUT_FOLDER_PATH} and run this cell again.")
else:
    predict_multiple_custom_images(INPUT_FOLDER_PATH, model, device, threshold=0.5)
